In [1]:
from pathlib import Path
from math import gcd, ceil, log2
from time import perf_counter
from urllib.request import urlretrieve
from contextlib import contextmanager
import json
import shutil
import sys
import warnings

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

ROOT = Path.cwd().resolve()
SPLIT = "test"
MAX_SAMPLES = None
SEED = 42
VIDEO_CONFIDENCE_THRESHOLD = 0.90
BATTERY_FRAME_CAPS = {1.00: 8, 0.70: 4, 0.40: 2, 0.20: 1}
VIDEO_CHECKPOINT = ROOT / "Audio/Video_checkpoints/Pruned_10/prune_0p1_full.pt"
DOWNLOAD_MISSING_VIDEOS = True
ARCHIVE_URL = "https://zenodo.org/records/15588220/files/mammalps_v1.zip"

AUDIO_DIR = ROOT / "Audio"
DATA_DIR = ROOT / "data"
MANIFEST_PATH = DATA_DIR / "metadata/benchmark_2_audio_model_manifest.csv"
AUDIO_CACHE = DATA_DIR / "audio_cache/full"
VIDEO_CACHE = DATA_DIR / "video_cache/raw"
OUT_DIR = ROOT / "results/pipeline_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_CACHE.mkdir(parents=True, exist_ok=True)

# All three models run on CPU for comparable local timing.
DEVICE = torch.device("cpu")
assert MANIFEST_PATH.is_file(), f"Benchmark II manifest not found: {MANIFEST_PATH}"
assert VIDEO_CHECKPOINT.is_file(), f"Trained video checkpoint not found: {VIDEO_CHECKPOINT}"
assert 0 <= VIDEO_CONFIDENCE_THRESHOLD <= 1
print("Project:", ROOT)
print("Device:", DEVICE)
print("Results:", OUT_DIR)

Project: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project
Device: cpu
Results: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\results\pipeline_comparison


In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
required = {"sample_id", "event_id", "file_id", "start_2s", "end_2s", "binary_label", "dataset_split", "remote_video_path"}
assert required.issubset(manifest), f"Missing manifest columns: {sorted(required - set(manifest))}"
assert manifest.sample_id.is_unique
assert manifest.groupby("event_id").dataset_split.nunique().max() == 1
selected = manifest.loc[manifest.dataset_split.eq(SPLIT)].copy().reset_index(drop=True)
assert np.allclose(selected.end_2s - selected.start_2s, 2.0, atol=0.002)

video_train_path = DATA_DIR / "metadata/train.csv"
if video_train_path.is_file():
    train_meta = pd.read_csv(video_train_path)
    train_file_ids = set(train_meta.video_path.astype(str).str.extract(r"(S\d+_C\d+_E\d+_V\d+)")[0].dropna())
    train_overlap = selected.file_id.isin(train_file_ids)
    print("Windows sharing a recording with video training:", int(train_overlap.sum()))
    assert not train_overlap.any(), "Evaluation recordings overlap video training recordings"
else:
    warnings.warn("Video-training recording overlap could not be checked: data/metadata/train.csv is absent")

if MAX_SAMPLES is not None:
    assert MAX_SAMPLES > 0
    minority = min(MAX_SAMPLES // 2, int((selected.binary_label == 0).sum()))
    backgrounds = selected[selected.binary_label.eq(0)].sample(n=minority, random_state=SEED)
    animal_count = min(MAX_SAMPLES - len(backgrounds), int((selected.binary_label == 1).sum()))
    animals = selected[selected.binary_label.eq(1)].sample(n=animal_count, random_state=SEED)
    selected = pd.concat([backgrounds, animals], ignore_index=True)
assert len(selected) > 0
missing_audio = sorted({str(AUDIO_CACHE / f"{fid}.flac") for fid in selected.file_id if not (AUDIO_CACHE / f"{fid}.flac").is_file()})
if missing_audio:
    raise FileNotFoundError(f"{len(missing_audio)} cached FLAC files are missing; first example: {missing_audio[0]}")
print("Windows:", len(selected), "| labels:", selected.binary_label.value_counts().to_dict())
display(selected[["sample_id", "file_id", "start_2s", "end_2s", "binary_label"]].head())

Windows sharing a recording with video training: 0
Windows: 207 | labels: {1: 123, 0: 84}


,sample_id,file_id,start_2s,end_2s,binary_label
0,E130_0,S1_C1_E130_V0579,0.300000,2.300000,1
1,E130_1,S1_C1_E130_V0581,0.333333,2.333333,1
2,E130_2,S1_C1_E130_V0585,0.316667,2.316667,1
3,E131_0,S1_C2_E131_V0289,0.383333,2.383333,1
4,E131_1,S1_C2_E131_V0296,1.983333,3.983333,1


In [3]:
LR_ARTIFACT = AUDIO_DIR / "models/mfcc_gate/mfcc_lr_2p0s.joblib"
assert LR_ARTIFACT.is_file(), f"Saved MFCC + LR model not found: {LR_ARTIFACT}"
lr_saved = joblib.load(LR_ARTIFACT)
LR_MODEL = lr_saved["model"]
LR_THRESHOLD = float(lr_saved["animal_threshold"])
fc = lr_saved["feature_config"]
assert fc.get("features") == "mfcc_mean_std"
assert float(fc["seconds"]) == 2.0
LR_SR = int(fc["sample_rate"])

def resample_to(wave, original_sr, target_sr):
    if original_sr == target_sr:
        return np.asarray(wave, dtype=np.float32)
    divisor = gcd(original_sr, target_sr)
    return resample_poly(wave, up=target_sr//divisor, down=original_sr//divisor).astype(np.float32)

def fixed_window(wave, target_sr):
    expected = round(2.0 * target_sr)
    if len(wave) < expected - round(0.05 * target_sr):
        raise ValueError("Audio window is shorter than 1.95 seconds")
    return np.clip(np.pad(wave, (0, max(0, expected-len(wave))))[:expected], -1, 1).astype(np.float32)

def lr_predict(audio, sr):
    wave = fixed_window(resample_to(audio, sr, LR_SR), LR_SR)
    mfcc = librosa.feature.mfcc(y=wave, sr=LR_SR, n_mfcc=int(fc["n_mfcc"]),
        n_mels=int(fc["n_mels"]), n_fft=int(fc["n_fft"]),
        hop_length=int(fc["hop_length"]), center=bool(fc["center"]))
    features = np.r_[mfcc.mean(axis=1), mfcc.std(axis=1)].astype(np.float32)
    probability = float(LR_MODEL.predict_proba(features[None, :])[0, 1])
    return {"probability": probability, "prediction": int(probability >= LR_THRESHOLD)}

print("MFCC + LR threshold:", round(LR_THRESHOLD, 6))

MFCC + LR threshold: 0.159498


In [4]:
MODEL_DIR = AUDIO_DIR / "models"
RESULTS_DIR = AUDIO_DIR / "results"
panns_paths = {
    "pretrained": MODEL_DIR / "pretrained/Cnn14_mAP=0.431.pth",
    "block2": MODEL_DIR / "panns_exit1_block2_2p0s.pt",
    "block4": MODEL_DIR / "panns_exit2_block4_2p0s.pt",
    "final": MODEL_DIR / "panns_cnn14_binary_head_2p0s.pt",
    "policy": RESULTS_DIR / "panns_early_exit_policy_results_2p0s.csv",
    "thresholds": RESULTS_DIR / "panns_early_exit_standalone_2p0s.csv",
    "flops": RESULTS_DIR / "panns_audio_all_paths_flops_2p0s.csv",
}
missing = [str(p) for p in panns_paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing PANNs artifacts:\n" + "\n".join(missing))

labels_path = Path.home() / "panns_data/class_labels_indices.csv"
if not labels_path.is_file():
    labels_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve("https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv", labels_path)
from panns_inference import AudioTagging

policies = pd.read_csv(panns_paths["policy"])
balanced = policies.loc[policies.operating_point.eq("balanced") & policies.split.eq("validation")]
assert len(balanced) == 1
M1, M2 = float(balanced.iloc[0].margin1), float(balanced.iloc[0].margin2)
threshold_data = pd.read_csv(panns_paths["thresholds"])
val_t = threshold_data.loc[threshold_data.split.eq("validation")]
def selected_threshold(term):
    matches = val_t.loc[val_t["exit"].str.lower().str.contains(term)]
    assert len(matches) == 1
    return float(matches.iloc[0].threshold)
T1, T2, TF = [selected_threshold(term) for term in ("block 2", "block 4", "final")]

panns_wrapper = AudioTagging(checkpoint_path=str(panns_paths["pretrained"]), device="cpu")
PANNS = panns_wrapper.model.module if isinstance(panns_wrapper.model, nn.DataParallel) else panns_wrapper.model
PANNS.eval().requires_grad_(False)

class BinaryHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.network = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim),
                                     nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        return self.network(x).squeeze(-1)

def read_head(path, weights_key):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    head = BinaryHead(saved["input_dim"], saved["hidden_dim"], saved["dropout"])
    head.load_state_dict(saved[weights_key])
    return head.eval().requires_grad_(False)

HEAD2 = read_head(panns_paths["block2"], "state_dict")
HEAD4 = read_head(panns_paths["block4"], "state_dict")
HEAD6 = read_head(panns_paths["final"], "head_state_dict")

def pool_features(x):
    x = x.mean(dim=3)
    return x.max(dim=2).values + x.mean(dim=2)

def score_margin(p, threshold):
    a, b = np.clip([p, threshold], 1e-6, 1-1e-6)
    return float(np.log(a/(1-a)) - np.log(b/(1-b)))

@torch.inference_mode()
def panns_predict(audio, sr):
    x = fixed_window(resample_to(audio, sr, 32000), 32000)
    wav = torch.from_numpy(x[None, :])
    m = PANNS
    x = m.logmel_extractor(m.spectrogram_extractor(wav))
    x = m.bn0(x.transpose(1,3)).transpose(1,3)
    x = m.conv_block1(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block2(x, pool_size=(2,2), pool_type="avg")
    p1 = float(torch.sigmoid(HEAD2(pool_features(x))).item())
    d1 = score_margin(p1, T1)
    if abs(d1) >= M1:
        return {"probability": p1, "prediction": int(d1 >= M1), "exit": 1}
    x = m.conv_block3(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block4(x, pool_size=(2,2), pool_type="avg")
    p2 = float(torch.sigmoid(HEAD4(pool_features(x))).item())
    d2 = score_margin(p2, T2)
    if abs(d2) >= M2:
        return {"probability": p2, "prediction": int(d2 >= M2), "exit": 2}
    x = m.conv_block5(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block6(x, pool_size=(1,1), pool_type="avg")
    x = x.mean(dim=3)
    emb = F.relu(m.fc1(x.max(dim=2).values + x.mean(dim=2)))
    p6 = float(torch.sigmoid(HEAD6(emb)).item())
    return {"probability": p6, "prediction": int(p6 >= TF), "exit": 3}

panns_ops = pd.read_csv(panns_paths["flops"])
panns_costs = dict(zip(panns_ops.path, panns_ops.gflops_est))
PANNS_GFLOPS = {1: panns_costs["adaptive_exit1"],
                2: panns_costs["adaptive_exit2"], 3: panns_costs["adaptive_final"]}
print("PANNs margins:", M1, M2, "| decision thresholds:", T1, T2, TF)

Checkpoint path: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\models\pretrained\Cnn14_mAP=0.431.pth


C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using CPU.
PANNs margins: 2.0 0.25 | decision thresholds: 0.2324143648147583 0.2000872790813446 0.1661750376224517


In [5]:
import importlib
import inspect

video_dir = ROOT / "Video"
assert (video_dir / "video.py").is_file(), f"Video implementation not found: {video_dir / 'video.py'}"
if str(video_dir) not in sys.path:
    sys.path.insert(0, str(video_dir))
importlib.invalidate_caches()
video_lib = importlib.import_module("video")
assert Path(video_lib.__file__).resolve() == (video_dir / "video.py").resolve(), "Python resolved a different video.py"
from torchvision.models import MobileNet_V3_Large_Weights
from fvcore.nn import FlopCountAnalysis

# The full PyTorch checkpoint contains a trusted serialized pruned model.
video_saved = torch.load(VIDEO_CHECKPOINT, map_location="cpu", weights_only=False)
assert video_saved.get("target") == "activity" and "model" in video_saved
assert video_saved.get("aggregation") == "mean_logits"
assert video_saved.get("preprocessing") == str(MobileNet_V3_Large_Weights.DEFAULT)
VIDEO = video_saved["model"].to(DEVICE).eval()
VIDEO.device = DEVICE
VIDEO_TRANSFORM = MobileNet_V3_Large_Weights.DEFAULT.transforms()
CLASS_NAMES = {int(v): k for k, v in video_saved["class_to_idx"].items()}
assert {"adaptive_threshold", "return_frames_used"}.issubset(inspect.signature(VIDEO.forward).parameters)
assert video_lib.nested_frame_positions(8)[:4] == [0.5, 0.0, 1.0, 0.25]

with torch.inference_mode():
    flop_profile = FlopCountAnalysis(VIDEO.model, torch.zeros(1,3,224,224,device=DEVICE))
    FRAME_GFLOPS = 2 * float(flop_profile.total()) / 1e9
    unsupported = dict(flop_profile.unsupported_ops())
if unsupported:
    print("Unsupported video profiler operators:", unsupported)
print("Video checkpoint:", VIDEO_CHECKPOINT.name)
print("Video cost per executed frame (estimated GFLOPs):", round(FRAME_GFLOPS, 5))

C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Unsupported operator aten::hardswish_ encountered 21 time(s)
Unsupported operator aten::add_ encountered 10 time(s)
Unsupported operator aten::hardsigmoid encountered 8 time(s)
Unsupported operator aten::mul encountered 8 time(s)
Unsupported operator aten::dropout_ encountered 1 time(s)


Unsupported video profiler operators: {'aten::hardswish_': 21, 'aten::add_': 10, 'aten::hardsigmoid': 8, 'aten::mul': 8, 'aten::dropout_': 1}
Video checkpoint: prune_0p1_full.pt
Video cost per executed frame (estimated GFLOPs): 0.36614


In [6]:
def read_audio(row):
    with sf.SoundFile(str(AUDIO_CACHE / (str(row["file_id"]) + ".flac"))) as f:
        sr = int(f.samplerate)
        f.seek(round(float(row["start_2s"]) * sr))
        x = f.read(frames=round(2.0*sr), dtype="float32", always_2d=False)
    if x.ndim == 2:
        x = x.mean(axis=1)
    return x.astype(np.float32), sr

# The first call initializes audio frontend kernels outside timed measurements.
warm_audio, warm_sr = read_audio(selected.iloc[0])
_ = lr_predict(warm_audio, warm_sr)
_ = panns_predict(warm_audio, warm_sr)

audio_rows = []
for i, (_, row) in enumerate(selected.iterrows(), start=1):
    t0 = perf_counter()
    wave, sr = read_audio(row)
    read_ms = (perf_counter() - t0) * 1000
    t0 = perf_counter()
    lr = lr_predict(wave, sr)
    lr_ms = (perf_counter() - t0) * 1000
    t0 = perf_counter()
    pn = panns_predict(wave, sr)
    panns_ms = (perf_counter() - t0) * 1000
    audio_rows.append({"sample_id": str(row.sample_id), "file_id": str(row.file_id),
        "binary_label": int(row.binary_label), "audio_read_ms": read_ms,
        "lr_probability": lr["probability"], "lr_prediction": lr["prediction"], "lr_ms": lr_ms,
        "panns_probability": pn["probability"], "panns_prediction": pn["prediction"],
        "panns_exit": pn["exit"], "panns_ms": panns_ms})
    if i % 25 == 0 or i == len(selected):
        print(f"Audio evaluated: {i}/{len(selected)}", flush=True)
audio_results = pd.DataFrame(audio_rows)
audio_results.to_csv(OUT_DIR / "audio_predictions.csv", index=False)

def gate_summary(predicted):
    positive = audio_results.binary_label.eq(1)
    negative = ~positive
    return {"animal_recall": float(audio_results.loc[positive, predicted].mean()),
            "background_rejection": float(1 - audio_results.loc[negative, predicted].mean()),
            "missed_animals": int(((audio_results[predicted]==0) & positive).sum()),
            "video_activation_rate": float(audio_results[predicted].mean())}

print("PANNs:", gate_summary("panns_prediction"))
print("MFCC + LR:", gate_summary("lr_prediction"))
display(audio_results.head())

Audio evaluated: 25/207
Audio evaluated: 50/207
Audio evaluated: 75/207
Audio evaluated: 100/207
Audio evaluated: 125/207
Audio evaluated: 150/207
Audio evaluated: 175/207
Audio evaluated: 200/207
Audio evaluated: 207/207
PANNs: {'animal_recall': 0.983739837398374, 'background_rejection': 0.8571428571428572, 'missed_animals': 2, 'video_activation_rate': 0.642512077294686}
MFCC + LR: {'animal_recall': 0.983739837398374, 'background_rejection': 0.8214285714285714, 'missed_animals': 2, 'video_activation_rate': 0.6570048309178744}


,sample_id,file_id,binary_label,audio_read_ms,lr_probability,lr_prediction,lr_ms,panns_probability,panns_prediction,panns_exit,panns_ms
0,E130_0,S1_C1_E130_V0579,1,2.1183,0.866076,1,6.2419,0.611098,1,2,18.8357
1,E130_1,S1_C1_E130_V0581,1,1.8790,0.976562,1,5.0256,0.614819,1,2,18.6624
2,E130_2,S1_C1_E130_V0585,1,1.8837,0.984415,1,5.9172,0.677423,1,2,18.5331
3,E131_0,S1_C2_E131_V0289,1,2.0804,0.997615,1,5.8382,0.934364,1,1,13.3675
4,E131_1,S1_C2_E131_V0296,1,2.0014,0.665033,1,5.7004,0.803291,1,1,11.4598


In [ ]:
import av

def sample_eight_frames(video_path, start_s, end_s):
    times = float(start_s) + (float(end_s) - float(start_s)) * np.asarray(video_lib.nested_frame_positions(8))
    nearest = [float("inf")] * 8
    images = [None] * 8
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for frame in container.decode(video=0):
            second = frame.time if frame.time is not None else (
                float(frame.pts * stream.time_base) if frame.pts is not None else None)
            if second is None:
                continue
            for j, time_point in enumerate(times):
                distance = abs(float(second) - time_point)
                if distance < nearest[j]:
                    nearest[j] = distance
                    images[j] = frame.to_ndarray(format="rgb24")
            if second > float(end_s) + 0.5 and all(x is not None for x in images):
                break
    if any(frame is None for frame in images) or max(nearest) > 0.5:
        raise RuntimeError(f"Video frames unavailable near the requested window: {video_path}")
    return torch.stack([VIDEO_TRANSFORM(torch.from_numpy(rgb).permute(2,0,1)) for rgb in images]).unsqueeze(0)

@contextmanager
def raw_video_archive():
    archive = None
    member_names = None
    try:
        def ensure(row):
            nonlocal archive, member_names
            destination = VIDEO_CACHE / (str(row["file_id"]) + ".mp4")
            if destination.is_file() and destination.stat().st_size:
                return destination, 0.0
            if not DOWNLOAD_MISSING_VIDEOS:
                raise FileNotFoundError(f"Uncached source video: {destination}")
            if archive is None:
                from remotezip import RemoteZip
                archive = RemoteZip(ARCHIVE_URL, timeout=120)
                member_names = [info.filename.replace("\\", "/") for info in archive.infolist() if not info.is_dir()]
            requested = str(row["remote_video_path"]).replace("\\", "/").lstrip("/")
            matches = [m for m in member_names if m == requested or m.endswith("/" + requested)]
            if len(matches) != 1:
                raise FileNotFoundError(f"Archive member ambiguous or absent: {requested} (matches={len(matches)})")
            partial = destination.with_suffix(".download")
            t0 = perf_counter()
            try:
                with archive.open(matches[0]) as source, partial.open("wb") as target:
                    shutil.copyfileobj(source, target, length=1024*1024)
                if partial.stat().st_size == 0:
                    raise IOError(f"Empty video: {requested}")
                partial.replace(destination)
            except Exception:
                partial.unlink(missing_ok=True)
                raise
            return destination, (perf_counter() - t0) * 1000
        yield ensure
    finally:
        if archive is not None:
            archive.close()

@torch.inference_mode()
def run_video(frames, cap):
    t0 = perf_counter()
    scores, counts = VIDEO(frames[:, :cap].to(DEVICE),
                            adaptive_threshold=VIDEO_CONFIDENCE_THRESHOLD,
                            return_frames_used=True)
    model_ms = (perf_counter() - t0) * 1000
    k = int(counts[0])
    conf = torch.softmax(scores[0], dim=-1)
    index = int(conf.argmax())
    return {"frames_used": k, "video_model_ms": model_ms,
            "activity_prediction": CLASS_NAMES[index], "activity_confidence": float(conf[index])}

import importlib.util
if DOWNLOAD_MISSING_VIDEOS and importlib.util.find_spec("remotezip") is None:
    raise ModuleNotFoundError("remotezip is required for selective raw-video downloads: python -m pip install remotezip")

with torch.inference_mode():
    _ = VIDEO.model(torch.zeros(1,3,224,224,device=DEVICE))
video_rows = []
with raw_video_archive() as ensure_video:
    for i, (_, row) in enumerate(selected.iterrows(), start=1):
        path, download_ms = ensure_video(row)
        t0 = perf_counter()
        tensor = sample_eight_frames(path, row.start_2s, row.end_2s)
        decode_ms = (perf_counter() - t0) * 1000
        for battery, cap in BATTERY_FRAME_CAPS.items():
            observed = run_video(tensor, cap)
            video_rows.append({"sample_id": str(row.sample_id), "battery": battery,
                               "frame_cap": cap, "video_decode_ms": decode_ms,
                               "download_ms": download_ms, **observed})
        if i % 10 == 0 or i == len(selected):
            pd.DataFrame(video_rows).to_csv(OUT_DIR / "video_predictions.csv", index=False)
            print(f"Video evaluated: {i}/{len(selected)}", flush=True)
video_results = pd.DataFrame(video_rows)
assert len(video_results) == len(selected) * len(BATTERY_FRAME_CAPS)
assert (video_results.frames_used <= video_results.frame_cap).all()
print("Video inference completed:", len(video_results), "battery-specific predictions")

Video evaluated: 10/207


In [ ]:
lr_cost_file = AUDIO_DIR / "results/mfcc_gate/mfcc_lr_rf_test_comparison.csv"
if lr_cost_file.is_file():
    lr_costs = pd.read_csv(lr_cost_file)
    lr_match = lr_costs[lr_costs.model.eq("MFCC + LR")]
    assert len(lr_match) == 1
    LR_GOPS_EST = float(lr_match.iloc[0].approx_frontend_gops)
else:
    raise FileNotFoundError(f"MFCC operation estimate not found: {lr_cost_file}")

joined = video_results.merge(audio_results, on="sample_id", validate="many_to_one")
joined["video_gflops"] = FRAME_GFLOPS * joined.frames_used
joined["video_local_ms"] = joined.video_decode_ms + joined.video_model_ms

strategies = []
for label, gate_col, gate_cost, gate_time in [
    ("Video only", None, None, None),
    ("PANNs + video", "panns_prediction", "panns", "panns_ms"),
    ("MFCC + LR + video", "lr_prediction", "lr", "lr_ms"),
]:
    z = joined.copy()
    active = np.ones(len(z), dtype=bool) if gate_col is None else z[gate_col].to_numpy(dtype=bool)
    if gate_cost == "panns":
        audio_ops = z.panns_exit.map(PANNS_GFLOPS).to_numpy(dtype=float)
    elif gate_cost == "lr":
        audio_ops = np.full(len(z), LR_GOPS_EST)
    else:
        audio_ops = np.zeros(len(z))
    if gate_col is None:
        audio_ms = np.zeros(len(z))
    else:
        audio_ms = (z.audio_read_ms + z[gate_time]).to_numpy(dtype=float)
    z["strategy"] = label
    z["activated"] = active
    z["effective_frames"] = np.where(active, z.frames_used, 0)
    z["audio_gops_est"] = audio_ops
    z["video_gops_est"] = np.where(active, z.video_gflops, 0.0)
    z["total_gops_est"] = z.audio_gops_est + z.video_gops_est
    z["local_latency_ms"] = audio_ms + np.where(active, z.video_local_ms, 0.0)
    z["activity_prediction"] = z.activity_prediction.where(active)
    strategies.append(z)
per_sample = pd.concat(strategies, ignore_index=True)
per_sample.to_csv(OUT_DIR / "pipeline_per_sample.csv", index=False)

summary = (per_sample.groupby(["battery", "frame_cap", "strategy"], as_index=False)
    .agg(windows=("sample_id", "size"), video_activation_rate=("activated", "mean"),
         avg_frames=("effective_frames", "mean"), avg_total_gops_est=("total_gops_est", "mean"),
         avg_local_latency_ms=("local_latency_ms", "mean")))

baseline = summary.loc[summary.strategy.eq("Video only"), ["battery", "avg_total_gops_est", "avg_local_latency_ms"]]
baseline = baseline.rename(columns={"avg_total_gops_est":"baseline_gops", "avg_local_latency_ms":"baseline_latency_ms"})
summary = summary.merge(baseline, on="battery", validate="many_to_one")
summary["compute_saving_vs_video_pct"] = 100 * (1 - summary.avg_total_gops_est / summary.baseline_gops)
summary["latency_saving_vs_video_pct"] = 100 * (1 - summary.avg_local_latency_ms / summary.baseline_latency_ms)
for key, col in [("PANNs + video", "panns_prediction"), ("MFCC + LR + video", "lr_prediction")]:
    gate = gate_summary(col)
    for label, value in gate.items():
        summary.loc[summary.strategy.eq(key), label] = value
summary.to_csv(OUT_DIR / "pipeline_summary.csv", index=False)
display(summary[["battery", "strategy", "video_activation_rate", "avg_frames", "avg_total_gops_est",
                 "compute_saving_vs_video_pct", "avg_local_latency_ms", "animal_recall", "background_rejection",
                 "missed_animals"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
order = ["Video only", "PANNs + video", "MFCC + LR + video"]
for label in order:
    part = summary.loc[summary.strategy.eq(label)].sort_values("battery")
    axes[0].plot(100*part.battery, part.avg_total_gops_est, marker="o", label=label)
    axes[1].plot(100*part.battery, part.avg_local_latency_ms, marker="o", label=label)
axes[0].set(xlabel="Simulated battery (%)", ylabel="Estimated GOPs/window", title="Average model computation")
axes[1].set(xlabel="Simulated battery (%)", ylabel="Local processing time (ms/window)", title="Measured-component latency")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.set_xticks(sorted(100*b for b in BATTERY_FRAME_CAPS))
axes[0].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "pipeline_cost_by_battery.png", dpi=180)
plt.show()

metadata = {"dataset_split":SPLIT, "samples":len(selected), "video_checkpoint":str(VIDEO_CHECKPOINT.relative_to(ROOT)),
            "video_confidence_threshold":VIDEO_CONFIDENCE_THRESHOLD,
            "battery_frame_caps":BATTERY_FRAME_CAPS,
            "lr_probability_threshold":LR_THRESHOLD,
            "panns_thresholds":{"block2":T1,"block4":T2,"final":TF},
            "panns_margins":{"block2":M1,"block4":M2},
            "video_frame_gflops_est":FRAME_GFLOPS, "lr_frontend_gops_heuristic":LR_GOPS_EST,
            "operation_method":"PANNs: THOP MAC×2; Video: fvcore count×2; MFCC: analytical proxy. Decoder and audio frontend not covered by profiler counts.",
            "latency_method":"Timed FLAC read + audio inference and timed raw-video decode + video inference, with conditional video execution reusing matched video-only results. Download excluded.",
            "activity_accuracy":"Not available for temporally unmatched Benchmark II activity labels",
            "energy":"No power or battery life measured"}
(OUT_DIR / "run_info.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", OUT_DIR)